# Figure: Equilibrium constants for homogeneous vapor reactions

In [ ]:
from pathlib import Path
import numpy as np

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [2]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_TICK_LEN,
    PLOTLY_FONT,
    PLOTLY_LEGEND_FONTSIZE,
    PLOTLY_TICK_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    TOOL_COLORS_HEX,
)
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "EVo", "MAGEC", "SulfurX", "VolFe", "VESIcal_Iacono"]

In [3]:
systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)
{sample: sorted(by_tool) for sample, by_tool in systems.items()}


{'MORB': ['DCompress', 'EVo', 'MAGEC', 'SulfurX', 'VESIcal_Iacono', 'VolFe'],
 'Kilauea': ['DCompress',
  'EVo',
  'MAGEC',
  'SulfurX',
  'VESIcal_Iacono',
  'VolFe'],
 'Fuego': ['DCompress', 'EVo', 'MAGEC', 'SulfurX', 'VESIcal_Iacono', 'VolFe'],
 'Fogo': ['DCompress', 'EVo', 'MAGEC', 'SulfurX', 'VESIcal_Iacono', 'VolFe']}

## Build the figure

In [4]:
# Row definitions: (DataFrame column, y-axis label).
Y_ROWS = [
    ("H","log<sub>10</sub>[K<sub>H</sub>]",1,1),
    ("C", "log<sub>10</sub>[K<sub>C</sub>]",1,2),
    ("S","log<sub>10</sub>[K<sub>S</sub>]",1,3),
    ('SH',"log<sub>10</sub>[K<sub>HS</sup>]",2,1),
    ("HC","log<sub>10</sub>[K<sub>CH</sup>]",2,2),
    ('CS',"log<sub>10</sub>[K<sub>SC</sup>]",2,3),
]

n_rows, n_cols = 2, 3
top_titles = [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
subplot_titles = top_titles + [""] * ((n_rows - 1) * n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    shared_xaxes=True, vertical_spacing=0.03, horizontal_spacing=0.08,
)

for r, (species,y_label,row,col) in enumerate(Y_ROWS, start=1):
    for c, sample in enumerate(SAMPLES, start=1):
        r = row
        c = col
        if sample == 'MORB':
            temperature = 1100.
        elif sample == 'Kilauea':
            temperature = 1220.
        elif sample == 'Fuego':
            temperature = 1030.
        elif sample == 'Fogo':
            temperature = 1200.
        size = 24
        for tool in TOOLS:
            size = size - 4
            if tool == 'VESIcal_Iacono':
                continue
            if tool == 'SulfurX':
                if species in ['H','C','S','HC','SC']:
                    continue
            if tool in ['DCompress','EVo']:
                if species == 'CS':
                    continue
            df = systems.get(sample, {}).get(tool)
            if df is None or "P_bars" not in df.columns:
                continue
            p_init = df["P_bars"].iloc[0]
            if p_init == 0:
                continue
            final = len(df)-1
            if df.loc[final,'P_bars'] > 1:
                continue
            if species == 'H':
                K = np.log10(df.loc[final,'H2O_v_mf']/(df.loc[final,'H2_v_mf']*((10.**df.loc[final,'logfO2'])**0.5)))
            elif species == 'C':
                K = np.log10(df.loc[final,'CO2_v_mf']/(df.loc[final,'CO_v_mf']*((10.**df.loc[final,'logfO2'])**0.5)))
            elif species == 'S':
                K = np.log10(df.loc[final,'SO2_v_mf']/(df.loc[final,'S2_v_mf']**0.5*((10.**df.loc[final,'logfO2']))))
            elif species == 'SH':
                K = np.log10((df.loc[final,'H2S_v_mf']*((10.**df.loc[final,'logfO2'])))/(df.loc[final,'SO2_v_mf']*df.loc[final,'H2_v_mf']))
            elif species == 'HC':
                K = np.log10((df.loc[final,'CO_v_mf']*df.loc[final,'H2_v_mf']**2.)/(df.loc[final,'CH4_v_mf']*(10**df.loc[final,'logfO2'])**0.5))
            elif species == 'CS':
                K = np.log10((df.loc[final,'CO_v_mf']*df.loc[final,'S2_v_mf']**0.5)/df.loc[final,'OCS_v_mf'])
            #x_norm = df["P_bars"] / p_init
            fig.add_trace(
                go.Scatter(
                    mode="markers",
                    x=[temperature], y=[K],
                    name=tool,
                    marker=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), size=size,
                                line_width =1, line_color = 'black'
                              ),
                    showlegend=(r == 1 and c == 1 and sample == 'Kilauea'),
                ),
                row=r, col=c,
            )
        fig.update_yaxes(title_text=y_label, row=r, col=c)
        if r == 2:
           fig.update_xaxes(title_text="T (\u00B0C)", row=r, col=c, range=[0, None])

legend_style_dict = dict(
    font=dict(size=PLOTLY_LEGEND_FONTSIZE),
    x=0.99, y=0.55,
    xanchor="right", yanchor="bottom",
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    height=500, width=1000,
    plot_bgcolor="white",
    margin=dict(t=40, r=30, l=60, b=50),
    font=PLOTLY_FONT,
    legend=legend_style_dict,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
    rangemode="tozero",
)

if SAVE_FIG:
    out = f"Fig_equilibrium_constants.png"

    # save to for_Notebook_Runs folder
    fig.write_image(f"{dirs_location}/figures/Fig_equilibrium_constants.png", scale=2, height=500, width=1000,)
    
    # also save to figures/ subfolder
    fig.write_image("figures/Fig_equilibrium_constants.png", scale=2, height=500, width=1000,)
    
    print(f"Saved {out}")

fig.show()

NameError: name 'SAVE_FIG' is not defined